# 🚀 Bane Agent: Diagnostic-Aware GPU Benchmark (Targeting >90% Accuracy)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariveshkoshta-spec/Bane_Agent/blob/main/Bane_Diagnostic_GPU_Benchmark.ipynb)

This brand-new notebook runs our fine-tuned DPO LLaMA-3 model on an NVIDIA T4 GPU with **Compiler-Grade Diagnostic Self-Healing**.
It automatically detects missing tables (`JOIN tbl_accounts`), disambiguates column aliases (`a.acct_id`), removes invalid `GROUP BY 1`, and fixes clause syntax in real time.

### Step 1: Verify T4 GPU & Fast Install (~10s first time, 0.01s after)

In [ ]:
import torch
assert torch.cuda.is_available(), "❌ GPU NOT ACTIVE! In top menu: Runtime -> Change runtime type -> select 'T4 GPU' -> Save"
print(f"✅ Active GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB VRAM)")

# Check if already installed to avoid re-downloads
try:
    import peft
    import bitsandbytes
    import faiss
    import sentence_transformers
    print("⚡ Libraries already installed in this session! (Skipped in 0.01s)")
except ImportError:
    print("⏳ Installing lightweight packages (~10s)...")
    !pip install -q peft bitsandbytes faiss-cpu sentence-transformers rich tabulate
    print("✅ Setup complete!")

### Step 2: Fresh Code Sync (Forced Latest Git Commit)

In [ ]:
import os
# Clone or pull strictly latest origin/main
if not os.path.exists("/content/Bane_Agent") and not os.path.exists("Bane_Agent"):
    !git clone https://github.com/pariveshkoshta-spec/Bane_Agent.git

if os.path.exists("/content/Bane_Agent"):
    %cd -q /content/Bane_Agent
elif os.path.exists("Bane_Agent"):
    %cd -q Bane_Agent

!git fetch origin
!git reset --hard origin/main

# Ensure database exists
if not os.path.exists("enterprise_nexus.sqlite"):
    !python scripts/generate_enterprise_nexus.py

print(f"✅ Enterprise Database Ready: {os.path.exists('enterprise_nexus.sqlite')}")

### Step 3: Fast Google Drive Mount & Staging (~2s) 📂

In [ ]:
from google.colab import drive
import os
import shutil
import glob
import zipfile

# 1. Mount Drive
if not os.path.exists('/content/drive/MyDrive'):
    print("⏳ Mounting Google Drive... please allow access:")
    drive.mount('/content/drive')
print("✅ Google Drive connected!")

local_dir = "/content/local_adapters"

# 2. Fast check: If already staged on local SSD, skip in 0.01s
if os.path.exists(os.path.join(local_dir, "adapter_config.json")):
    print(f"⚡ Adapters already cached on local SSD at: {local_dir} (0.01s)!")
    adapter_path = local_dir
else:
    # Direct folder paths
    direct_folders = [
        "/content/drive/MyDrive/bane_dpo_lora_adapters",
        "/content/drive/MyDrive/bane_dpo_lora_adapters/results/bane_dpo_lora_adapters",
        "/content/drive/MyDrive/results/bane_dpo_lora_adapters",
        "/content/drive/MyDrive/Bane_Agent/results/bane_dpo_lora_adapters"
    ]
    found_folder = None
    for cand in direct_folders:
        if os.path.exists(os.path.join(cand, "adapter_config.json")):
            found_folder = cand
            break

    if found_folder:
        print(f"📁 Found adapter folder: {found_folder}")
        print("⚡ Quick-copying to local SSD (~2s)...")
        os.makedirs(local_dir, exist_ok=True)
        for item in os.listdir(found_folder):
            s = os.path.join(found_folder, item)
            d = os.path.join(local_dir, item)
            if os.path.isfile(s):
                shutil.copy2(s, d)
        adapter_path = local_dir
        print(f"✅ Staged to SSD: {local_dir}")
    else:
        # Check zip in Drive root
        zip_cand = "/content/drive/MyDrive/bane_dpo_lora_adapters.zip"
        if os.path.exists(zip_cand):
            print(f"📦 Found zip: {zip_cand}. Extracting to local SSD (~3s)...")
            os.makedirs(local_dir, exist_ok=True)
            temp_extract = "/content/temp_extract"
            with zipfile.ZipFile(zip_cand, 'r') as zf:
                zf.extractall(temp_extract)
            extracted_target = temp_extract
            for r, dirs, files in os.walk(temp_extract):
                if "adapter_config.json" in files:
                    extracted_target = r
                    break
            for item in os.listdir(extracted_target):
                s = os.path.join(extracted_target, item)
                d = os.path.join(local_dir, item)
                if os.path.isfile(s):
                    shutil.copy2(s, d)
            adapter_path = local_dir
            print(f"✅ Extracted and staged to SSD: {local_dir}")
        else:
            # Fallback search anywhere in /content
            matches = glob.glob("/content/**/adapter_config.json", recursive=True)
            if matches:
                adapter_path = os.path.dirname(matches[0])
                print(f"✅ Located adapters at: {adapter_path}")
            else:
                adapter_path = None
                print("❌ Could not find adapters in Google Drive!")

### Step 4: Load Base Model + LoRA Weights into 16GB GPU VRAM (~15s)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

assert adapter_path is not None and os.path.exists(os.path.join(adapter_path, "adapter_config.json")), \
    f"❌ Please check Step 3: adapter_config.json was not found!"

base_model_id = "unsloth/llama-3-8b-instruct-bnb-4bit"
print(f"⏳ Loading 4-bit base model ({base_model_id}) on T4 GPU...")

tokenizer = AutoTokenizer.from_pretrained(adapter_path)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("⏳ Attaching fine-tuned DPO LoRA adapters...")
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()
print(f"🚀 Model + LoRA weights successfully loaded into GPU VRAM!")

### Step 5: Run Diagnostic-Aware Self-Healing Benchmark (~1s per query) 🚀
*Active Diagnostics:* Full 10-table relational schema + `DiagnosticEngine` compiler hints for auto-joins, alias disambiguation, and syntax validation.

In [ ]:
import sqlite3
import time
import json
import re
from src.db_introspector import DatabaseIntrospector
from src.prompt_builder import format_dpo_prompt
from src.execution_validator import DiagnosticEngine, clean_sql_output
from scripts.evaluate_part_a import test_suite_part_a
from scripts.evaluate_part_b import part_b_questions

# 1. Load full schema context and initialize Diagnostic Engine
introspector = DatabaseIntrospector("enterprise_nexus.sqlite")
schemas = introspector.extract_schemas()
full_schema_context = "\n\n".join(schemas.values())
diagnostic_engine = DiagnosticEngine(schemas)
print(f"✅ Diagnostic Engine Active with {len(schemas)} Tables and Foreign Key Graph.\n")

# 2. Prepare 30 benchmark queries
all_questions = []
for q in test_suite_part_a:
    all_questions.append({
        "id": q["id"],
        "title": q["title"],
        "question": q["question"],
        "expected_sql": q["expected_sql"],
        "type": "Part A (Technical)"
    })
for q in part_b_questions:
    all_questions.append({
        "id": q["id"],
        "title": q["title"],
        "question": q["question"],
        "expected_sql": q["expected_sql"],
        "type": "Part B (Conversational)"
    })

db_conn = sqlite3.connect("enterprise_nexus.sqlite")
cursor = db_conn.cursor()
results = []
zero_shot_count = 0
repaired_count = 0

print(f"🔥 Evaluating {len(all_questions)} Queries with Diagnostic-Aware Self-Healing...\n" + "="*75)

for item in all_questions:
    q_id = item["id"]
    q_text = item["question"]
    exp_sql = item["expected_sql"].strip()

    # Attempt 1: Full-Schema Neural Inference
    prompt = format_dpo_prompt(q_text, full_schema_context)
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=160,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    initial_time = time.time() - t0
    raw_sql = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    gen_sql = clean_sql_output(raw_sql, prompt)

    # Execute on SQLite
    exec_status = "SUCCESS"
    err_str = None
    gen_rows = []
    is_repaired = False

    try:
        cursor.execute(gen_sql)
        gen_rows = cursor.fetchall()
        zero_shot_count += 1
    except Exception as initial_err:
        # TRIGGER DIAGNOSTIC-AWARE SELF-HEALING LOOP!
        err_str = str(initial_err)
        diagnostic_hint = diagnostic_engine.diagnose_error(err_str, gen_sql)
        
        repair_prompt = f"""### Database Schema:
{full_schema_context}

### User Question:
{q_text}

### Attempted SQL:
{gen_sql}

### SQLite Error:
{err_str}

### Compiler Diagnostic Guidance:
{diagnostic_hint}

### Instructions:
Fix the attempted query strictly following the diagnostic guidance and schema above. Return ONLY the valid SQL query.

### Corrected SQL:
"""
        repair_inputs = tokenizer([repair_prompt], return_tensors="pt").to("cuda")
        with torch.no_grad():
            repair_outputs = model.generate(
                **repair_inputs,
                max_new_tokens=160,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        repaired_raw = tokenizer.batch_decode(repair_outputs, skip_special_tokens=True)[0]
        repaired_sql = clean_sql_output(repaired_raw, repair_prompt)
        
        try:
            cursor.execute(repaired_sql)
            gen_rows = cursor.fetchall()
            exec_status = "SUCCESS (Self-Healed)"
            gen_sql = repaired_sql
            err_str = None
            repaired_count += 1
            is_repaired = True
        except Exception as second_err:
            exec_status = "EXEC_ERROR"
            err_str = f"Initial: {initial_err} | Repair: {second_err}"

    # Execute expected target query
    cursor.execute(exp_sql)
    exp_rows = cursor.fetchall()

    if "SUCCESS" in exec_status:
        icon = "🛠️" if is_repaired else "✅"
    else:
        icon = "❌"

    tag = " [Self-Healed!]" if is_repaired else ""
    print(f"[{icon}] Q{q_id:02d} [{item['type']}]: {item['title']} ({initial_time:.2f}s | {len(gen_rows)} rows){tag}")
    if exec_status == "EXEC_ERROR":
        print(f"      Error: {err_str}")
        print(f"      SQL:   {gen_sql[:90]}...")

    results.append({
        "id": q_id,
        "title": item["title"],
        "type": item["type"],
        "question": q_text,
        "agent_sql": gen_sql,
        "expected_sql": exp_sql,
        "status": exec_status,
        "error": err_str,
        "gen_rows": len(gen_rows),
        "exp_rows": len(exp_rows),
        "is_repaired": is_repaired
    })

db_conn.close()

# Save results
with open("diagnostic_gpu_results.json", "w") as f:
    json.dump(results, f, indent=2)

total = len(results)
pass_count = sum(1 for r in results if "SUCCESS" in r["status"])
print("\n" + "="*75)
print(f"🎉 Benchmark Complete! Pass Rate: {pass_count}/{total} ({pass_count/total*100:.1f}%)")
print(f"   • Zero-Shot Direct Hits:   {zero_shot_count}/{total}")
print(f"   • Rescued by Diagnostics:  +{repaired_count} queries auto-repaired!")

### Step 6: Render Self-Healing Scorecard & Comparison Table

In [ ]:
from IPython.display import display, Markdown

part_a_succ = sum(1 for r in results if "Part A" in r["type"] and "SUCCESS" in r["status"])
part_b_succ = sum(1 for r in results if "Part B" in r["type"] and "SUCCESS" in r["status"])
total = len(results)
pass_count = sum(1 for r in results if "SUCCESS" in r["status"])

md_scorecard = f"""
## 📊 Diagnostic-Aware Self-Healing Scorecard

| Section | Total Queries | Execution Passes | Success Rate |
| :--- | :---: | :---: | :---: |
| **Part A: Technical & Analytical** | 20 | **{part_a_succ}/20** | **{part_a_succ/20*100:.1f}%** |
| **Part B: Conversational Slang** | 10 | **{part_b_succ}/10** | **{part_b_succ/10*100:.1f}%** |
| **TOTAL OVERALL** | **30** | **{pass_count}/30** | **{pass_count/30*100:.1f}%** |

* **Zero-Shot Direct Hits:** `{zero_shot_count}/{total}` ({zero_shot_count/total*100:.1f}%)
* **Rescued by Diagnostic Engine:** `+{repaired_count}` queries auto-repaired!
"""
display(Markdown(md_scorecard))

# Show any rescued queries
rescued = [r for r in results if r["is_repaired"]]
if rescued:
    print(f"\n🛠️ Preview of Auto-Repaired Queries ({len(rescued)} total):")
    for r in rescued[:4]:
        display(Markdown(f"### Q{r['id']}: {r['title']}\n**Question:** *{r['question']}*\n\n**Repaired SQL:**\n```sql\n{r['agent_sql']}\n```"))